In [22]:
import pandas as pd
import numpy as np

raw_data = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
raw_data.shape

(7043, 21)

Replace empty strings with NaN then drop

In [23]:

raw_data['TotalCharges'] = raw_data['TotalCharges'].replace(' ', np.nan)
raw_data['TotalCharges'] = pd.to_numeric(raw_data['TotalCharges'])
raw_data = raw_data.dropna(subset=['TotalCharges'])

raw_data.shape  

(7032, 21)

In [24]:
raw_data = raw_data.drop(columns=['customerID'])

In [25]:
raw_data.shape

(7032, 20)

In [27]:
raw_data['Churn'] = raw_data['Churn'].map({'Yes': 1, 'No': 0})
raw_data['Churn'].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [28]:
cat_cols = raw_data.select_dtypes(include='object').columns.tolist()
num_cols = raw_data.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols.remove('Churn') 

print("Categorical:", cat_cols)
print("Numerical:", num_cols)

Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


One Hot Encoding

In [31]:
encoded_data = pd.get_dummies(raw_data, columns=cat_cols, drop_first=True)
encoded_data.shape 
encoded_data.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [32]:
encoded_data.shape

(7032, 31)

Split Feature and Target

In [33]:
X = encoded_data.drop(columns=['Churn'])
y = encoded_data['Churn']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Churn rate:", y.mean().round(3))

X shape: (7032, 30)
y shape: (7032,)
Churn rate: 0.266


Train and Test Split

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y      # important! maintains churn ratio in both splits
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("Train churn rate:", y_train.mean().round(3))
print("Test churn rate:", y_test.mean().round(3))

Train size: (5625, 30)
Test size: (1407, 30)
Train churn rate: 0.266
Test churn rate: 0.266


Scaling Numerical Columns

In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

Data Saving

In [36]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)